In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import DBSCAN
# from scipy.spatial.distance import cdist
# import os
# import warnings

# # غیرفعال کردن هشدارهای غیرضروری
# warnings.filterwarnings('ignore')

# # ۱. تنظیمات مسیرها و متغیرها
# file_path = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_lubrication_system_univariate\univariate\dsas_g11_univariate_output1.xlsx'

# os.makedirs(os.path.dirname(output_filename), exist_ok=True)

# # لیست تمام سنسورها برای تحلیل شبکه ارتباطی
# all_features = ['AssetID_9343','AssetID_9375', 'AssetID_8341', 'AssetID_8342', 'AssetID_8343', 
#                 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

# # سنسورهایی که تست ۳-سیگما روی آن‌ها اجرا می‌شود
# target_sensors = ['AssetID_9343','AssetID_9357','AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

# # ۲. خواندن داده‌ها
# try:
#     print("STEP 1")
#     df_raw = pd.read_excel(file_path)
#     print("STEP 2")
#     df_raw['date'] = pd.to_datetime(df_raw['date'])
#     print("STEP 3")
#     df_raw = df_raw.sort_values(by='date')
#     print("✅ دیتا با موفقیت بارگذاری شد.")
# except Exception as e:
#     print(f"❌ خطا در خواندن فایل: {e}")
#     exit()

# # ۳. پیش‌پردازش و حذف داده‌های پرت (DBSCAN - حذف ۱۰ درصد داده‌های دور)
# scaler = StandardScaler()
# print("STEP 4")
# scaled_data = scaler.fit_transform(df_raw[all_features])

# # خوشه‌بندی برای شناسایی نقاط پرت
# dbscan = DBSCAN(eps=0.5, min_samples=5)
# print("DBSCAN START...")
# labels = dbscan.fit_predict(scaled_data)
# print("DBSCAN END")


# cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

# def calculate_distance(i):
#     label, point = labels[i], scaled_data[i].reshape(1, -1)
#     if label != -1:
#         return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
#     return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

# df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]

# # حذف ۱۰ درصد داده‌های با بیشترین فاصله از خوشه‌ها
# df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
# df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# # ۴. جداسازی بازه نرمال (Baseline) و بازه تحلیل (Fault)
# last_date = df_cleaned.index.max()
# start_analysis_date = last_date - pd.Timedelta(days=30)
# baseline_end = start_analysis_date - pd.Timedelta(days=1)
# baseline_start = baseline_end - pd.Timedelta(days=30)

# df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)]
# df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()

# # ۵. تست چندسطحی انحراف معیار (3-Sigma Rule)
# def get_sigma_status(val, mean, std):
#     if std == 0: return 'Normal'
#     deviation = abs(val - mean) / std
#     if deviation > 3:
#         return 'Action Required'
#     elif deviation > 2:
#         return 'Warning'
#     elif deviation > 1:
#         return 'Normal (Minor Change)'
#     else:
#         return 'Normal'

# for col in target_sensors:
#     m = df_baseline[col].mean()
#     s = df_baseline[col].std() if df_baseline[col].std() != 0 else 1e-6
#     df_fault[f'Status_{col}'] = df_fault[col].apply(lambda x: get_sigma_status(x, m, s))

# # ۶. محاسبات همبستگی جزئی (Partial Correlation)
# def get_partial_corr(data, columns):

#     corr_matrix = data[columns].corr().values
#     # Regularization برای جلوگیری از خطای ماتریس تکین
#     precision = np.linalg.inv(corr_matrix + np.eye(corr_matrix.shape[0])*1e-6)
#     d = np.sqrt(np.diag(precision))
#     partial_corr = -precision / np.outer(d, d)
#     np.fill_diagonal(partial_corr, 1.0)
#     return pd.DataFrame(partial_corr, index=columns, columns=columns)

# pcorr_baseline = get_partial_corr(df_baseline, all_features)
# pcorr_fault = get_partial_corr(df_fault[all_features], all_features)
# delta_pcorr = pcorr_fault - pcorr_baseline

# # ۷. محاسبه امتیازات RCA و رتبه‌بندی ریشه خطا
# deviation_scores = {c: abs((df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() or 1e-6)) for c in all_features}
# change_scores = {c: np.mean([abs(delta_pcorr.loc[c, o]) for o in all_features if o != c]) for c in all_features}

# def normalize_dict(d):
#     vals = np.array(list(d.values()))
#     if vals.max() == vals.min(): return {k: 0.5 for k in d.keys()}
#     return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in d.items()}

# dev_norm = normalize_dict(deviation_scores)
# chg_norm = normalize_dict(change_scores)

# rca_list = []
# for c in all_features:
#     strength_fault = np.mean([abs(pcorr_fault.loc[c, o]) for o in all_features if o != c])
#     # فرمول تجمیعی RCA Score
#     score = (dev_norm[c]*0.5) + (chg_norm[c]*0.5)

#     rca_list.append({
#         'Sensor': c,
#         'Change_Normalized': chg_norm[c],
#         'Change_Score_Delta': change_scores[c],
#         'Deviation_Normalized': dev_norm[c],
#         'Deviation_Score': deviation_scores[c],
#         'Strength_Score_Fault': strength_fault,
#         'RCA_Score': score,
#         'Root_Cause_Suspicion': score*(1 - strength_fault)
#     })

# df_rca_summary = pd.DataFrame(rca_list).sort_values(by='Root_Cause_Suspicion', ascending=False)

# # ۸. ذخیره‌سازی در اکسل با شیت‌های مجزا
# try:
#     with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
#         # شیت ۱: جزئیات عیب‌یابی تک‌متغیره (Status)
#         status_cols = [f'Status_{c}' for c in target_sensors]
#         df_fault[target_sensors + status_cols].reset_index().to_excel(writer, sheet_name='Fault_Status', index=False)

#         # شیت ۲: خلاصه تحلیل ریشه خطا
#         df_rca_summary.to_excel(writer, sheet_name='RCA_Summary', index=False)

#         # شیت ۳: ماتریس تغییرات همبستگی
#         delta_pcorr.to_excel(writer, sheet_name='Delta_Correlation_Matrix')

#     print(f"🚀 تحلیل با موفقیت پایان یافت. خروجی: {output_filename}")
#     print("\n--- ۵ مظنون اصلی خرابی ---")
#     print(df_rca_summary[['Sensor', 'RCA_Score', 'Root_Cause_Suspicion']].head(5))

# except Exception as e:
#     print(f"❌ خطا در ذخیره فایل: {e}")

STEP 1
STEP 2
STEP 3
✅ دیتا با موفقیت بارگذاری شد.
STEP 4
DBSCAN START...
DBSCAN END
🚀 تحلیل با موفقیت پایان یافت. خروجی: outputs\G11\dsas_g11_lubrication_system_univariate\univariate\dsas_g11_univariate_output1.xlsx

--- ۵ مظنون اصلی خرابی ---
         Sensor  RCA_Score  Root_Cause_Suspicion
0  AssetID_9343        NaN                   NaN
1  AssetID_9375        NaN                   NaN
2  AssetID_8341        NaN                   NaN
3  AssetID_8342        NaN                   NaN
4  AssetID_8343        NaN                   NaN


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
import os
import warnings
import time
from datetime import datetime

# غیرفعال کردن هشدارهای غیرضروری
warnings.filterwarnings('ignore')

def run_analysis():
    """اجرای تحلیل و ذخیره خروجی"""
    
    print("="*60)
    print(f"🔄 شروع تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*60)
    
    # ۱. تنظیمات مسیرها و متغیرها
    file_path = r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx'
    output_filename = r'outputs\G11\dsas_g11_lubrication_system_univariate\univariate\dsas_g11_univariate_output1.xlsx'

    os.makedirs(os.path.dirname(output_filename), exist_ok=True)

    # لیست تمام سنسورها برای تحلیل شبکه ارتباطی
    all_features = ['AssetID_9343','AssetID_9375', 'AssetID_8341', 'AssetID_8342', 'AssetID_8343', 
                    'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

    # سنسورهایی که تست ۳-سیگما روی آن‌ها اجرا می‌شود
    target_sensors = ['AssetID_9343','AssetID_9357','AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287']

    # ۲. خواندن داده‌ها
    try:
        print("STEP 1")
        df_raw = pd.read_excel(file_path)
        print("STEP 2")
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        print("STEP 3")
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ دیتا با موفقیت بارگذاری شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return None

    # ۳. پیش‌پردازش و حذف داده‌های پرت (DBSCAN - حذف ۱۰ درصد داده‌های دور)
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    print("STEP 4")
    scaled_data = scaler.fit_transform(df_raw[all_features])

    # خوشه‌بندی برای شناسایی نقاط پرت
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    print("DBSCAN START...")
    labels = dbscan.fit_predict(scaled_data)
    print("DBSCAN END")

    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]

    # حذف ۱۰ درصد داده‌های با بیشترین فاصله از خوشه‌ها
    before_count = len(df_raw)
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")

    # ۴. جداسازی بازه نرمال (Baseline) و بازه تحلیل (Fault)
    print("🔄 مرحله 2: جداسازی بازه‌های نرمال و تحلیل...")
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    baseline_end = start_analysis_date - pd.Timedelta(days=1)
    baseline_start = baseline_end - pd.Timedelta(days=30)

    df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)]
    df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()

    print(f"   بازه نرمال: {baseline_start} تا {baseline_end}")
    print(f"   بازه تحلیل: {start_analysis_date} تا {last_date}")
    print(f"   تعداد رکوردهای بازه نرمال: {len(df_baseline):,}")
    print(f"   تعداد رکوردهای بازه تحلیل: {len(df_fault):,}")

    # ۵. تست چندسطحی انحراف معیار (3-Sigma Rule)
    print("🔄 مرحله 3: اجرای تست 3-Sigma...")
    
    def get_sigma_status(val, mean, std):
        if std == 0: return 'Normal'
        deviation = abs(val - mean) / std
        if deviation > 3:
            return 'Action Required'
        elif deviation > 2:
            return 'Warning'
        elif deviation > 1:
            return 'Normal (Minor Change)'
        else:
            return 'Normal'

    for col in target_sensors:
        if col in df_baseline.columns:
            m = df_baseline[col].mean()
            s = df_baseline[col].std() if df_baseline[col].std() != 0 else 1e-6
            df_fault[f'Status_{col}'] = df_fault[col].apply(lambda x: get_sigma_status(x, m, s))
    
    # نمایش خلاصه وضعیت‌ها
    for col in target_sensors:
        if col in df_fault.columns:
            status_counts = df_fault[f'Status_{col}'].value_counts()
            print(f"   {col}: {dict(status_counts)}")

    # ۶. محاسبات همبستگی جزئی (Partial Correlation)
    print("🔄 مرحله 4: محاسبات همبستگی جزئی...")
    
    def get_partial_corr(data, columns):
        corr_matrix = data[columns].corr().values
        # Regularization برای جلوگیری از خطای ماتریس تکین
        precision = np.linalg.inv(corr_matrix + np.eye(corr_matrix.shape[0])*1e-6)
        d = np.sqrt(np.diag(precision))
        partial_corr = -precision / np.outer(d, d)
        np.fill_diagonal(partial_corr, 1.0)
        return pd.DataFrame(partial_corr, index=columns, columns=columns)

    pcorr_baseline = get_partial_corr(df_baseline, all_features)
    pcorr_fault = get_partial_corr(df_fault[all_features], all_features)
    delta_pcorr = pcorr_fault - pcorr_baseline
    
    print("   ✅ محاسبات همبستگی انجام شد")

    # ۷. محاسبه امتیازات RCA و رتبه‌بندی ریشه خطا
    print("🔄 مرحله 5: محاسبه امتیازات RCA...")
    
    deviation_scores = {c: abs((df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() or 1e-6)) for c in all_features}
    change_scores = {c: np.mean([abs(delta_pcorr.loc[c, o]) for o in all_features if o != c]) for c in all_features}

    def normalize_dict(d):
        vals = np.array(list(d.values()))
        if vals.max() == vals.min(): return {k: 0.5 for k in d.keys()}
        return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in d.items()}

    dev_norm = normalize_dict(deviation_scores)
    chg_norm = normalize_dict(change_scores)

    rca_list = []
    for c in all_features:
        strength_fault = np.mean([abs(pcorr_fault.loc[c, o]) for o in all_features if o != c])
        # فرمول تجمیعی RCA Score
        score = (dev_norm[c]*0.5) + (chg_norm[c]*0.5)

        rca_list.append({
            'Sensor': c,
            'Change_Normalized': chg_norm[c],
            'Change_Score_Delta': change_scores[c],
            'Deviation_Normalized': dev_norm[c],
            'Deviation_Score': deviation_scores[c],
            'Strength_Score_Fault': strength_fault,
            'RCA_Score': score,
            'Root_Cause_Suspicion': score*(1 - strength_fault)
        })

    df_rca_summary = pd.DataFrame(rca_list).sort_values(by='Root_Cause_Suspicion', ascending=False)

    # ۸. ذخیره‌سازی در اکسل با شیت‌های مجزا
    print("💾 مرحله 6: ذخیره خروجی...")
    
    try:
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            # شیت ۱: جزئیات عیب‌یابی تک‌متغیره (Status)
            status_cols = [f'Status_{c}' for c in target_sensors if f'Status_{c}' in df_fault.columns]
            cols_to_save = [c for c in target_sensors if c in df_fault.columns] + status_cols
            if cols_to_save:
                df_fault[cols_to_save].reset_index().to_excel(writer, sheet_name='Fault_Status', index=False)

            # شیت ۲: خلاصه تحلیل ریشه خطا
            df_rca_summary.to_excel(writer, sheet_name='RCA_Summary', index=False)

            # شیت ۳: ماتریس تغییرات همبستگی
            delta_pcorr.to_excel(writer, sheet_name='Delta_Correlation_Matrix')

        print(f"✅ فایل با موفقیت ذخیره شد: {output_filename}")
        print(f"📊 تعداد رکوردهای نهایی: {len(df_fault):,}")
        print(f"📋 تعداد ستون‌ها: {len(df_fault.columns)}")
        
        print("\n--- ۵ مظنون اصلی خرابی ---")
        print(df_rca_summary[['Sensor', 'RCA_Score', 'Root_Cause_Suspicion']].head(5))
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return None
    
    print("="*60)
    print(f"✅ تحلیل در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} کامل شد")
    print("="*60)
    
    return df_fault, df_rca_summary

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز)
    """
    print("="*60)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل سیستم روغن‌کاری")
    print("="*60)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 10:00")
    print("   - ساعت 10:05")
    print("   - ساعت 10:10")
    print("="*60)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*60)
    
    last_run_time = None  # فقط برای جلوگیری از اجرای مجدد در یک زمان
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص
            if current_time in ["21:42", "21:43", "21:44"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_time != current_time:
                    print("\n" + "="*60)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*60)
                    
                    # اجرای تابع اصلی
                    result = run_analysis()
                    
                    if result is not None:
                        print("\n" + "="*60)
                        print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                        print("="*60)
                    else:
                        print("\n" + "="*60)
                        print("❌ اجرای زمان‌بندی شده با شکست مواجه شد!")
                        print("="*60)
                    
                    # ثبت زمان اجرا
                    last_run_time = current_time
                    
                    # 10 ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(10)
            
            # هر 10 ثانیه یکبار بررسی کن
            time.sleep(10)
            
        except KeyboardInterrupt:
            print("\n" + "="*60)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*60)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)

# اجرای اصلی
if __name__ == "__main__":
    try:
        print("="*60)
        print("🚀 شروع برنامه تحلیل سیستم روغن‌کاری")
        print("="*60)
        
        # شروع زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه تحلیل سیستم روغن‌کاری
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل سیستم روغن‌کاری
⏰ زمان‌های اجرا (هر روز):
   - ساعت 10:00
   - ساعت 10:05
   - ساعت 10:10
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-01 21:42:01
🔄 شروع تحلیل در 2026-07-01 21:42:01
STEP 1
STEP 2
STEP 3
✅ دیتا با موفقیت بارگذاری شد. تعداد رکوردها: 6,796
📅 بازه زمانی: 2021-03-20 17:37:26 تا 2026-05-26 12:30:11
🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...
STEP 4
DBSCAN START...
DBSCAN END
   حذف 679 ردیف به عنوان داده‌های پرت
🔄 مرحله 2: جداسازی بازه‌های نرمال و تحلیل...
   بازه نرمال: 2026-03-25 20:45:54 تا 2026-04-24 20:45:54
   بازه تحلیل: 2026-04-25 20:45:54 تا 2026-05-25 20:45:54
   تعداد رکوردهای بازه نرمال: 52
   تعداد رکوردهای بازه تحلیل: 96
🔄 مرحله 3: اجرای تست 3-Sigma...
   AssetID_9343: {'Normal': 81, 'Normal (Minor Change)': 14, 'Warning': 1}
   AssetID_9357: {'Normal': 30, 'Warning': 26, 'Normal (Minor Change)': 21, 'Action Required': 19}
   AssetID_9375: {'Normal': 75,